# NB0 — Environment, paths and configuration

Checks the environment, verifies the data against `data/MANIFEST_sha256.csv`, and writes `results/nb0_setup/` (config snapshot, environment, summary). Runs locally or on Colab.

On Colab: set `DRIVE_REPO` below to the Drive mirror of the repository before running.

In [1]:
import os, sys, json, time, subprocess, importlib
from pathlib import Path
ON_COLAB = 'google.colab' in sys.modules
DRIVE_REPO = '/content/drive/MyDrive/pv-cold-start'
if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['PVCOLD_ROOT'] = DRIVE_REPO
    ROOT = Path(DRIVE_REPO)
else:
    ROOT = Path.cwd().resolve()
    ROOT = ROOT.parent if ROOT.name == 'notebooks' else ROOT
sys.path.insert(0, str(ROOT / 'src'))
from pvcold import config as C, audit as A
t0 = time.time()
print('ROOT =', C.ROOT, '| on Colab:', ON_COLAB)

ROOT = C:\Users\Dell\Desktop\phd_ehtp\pv paper | on Colab: False


## Versions
LightGBM must be 4.6.0 (4.7.0 crashed on the author's Windows machine). The foundation-model packages are only needed on Colab (NB5b).

In [2]:
env = A.versions()
env['on_colab'] = ON_COLAB
for pkg in ['chronos-forecasting', 'timesfm', 'tabpfn-time-series', 'tabpfn']:
    try:
        from importlib.metadata import version, metadata
        env[pkg] = {'version': version(pkg), 'license': metadata(pkg).get('License')}
    except Exception as e:
        env[pkg] = f'not installed ({type(e).__name__})'
try:
    import torch
    env['cuda'] = torch.cuda.is_available()
    env['gpu'] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
except Exception:
    env['cuda'] = False
assert env['lightgbm'] == '4.6.0', f"lightgbm {env['lightgbm']} (expected 4.6.0)"
for k, v in env.items():
    print(f'{k:22s} {v}')

python                 3.12.1
platform               Windows-11-10.0.26300-SP0
executable             C:\Python312\python.exe
run_at                 2026-11-18 23:01:50
numpy                  1.26.4
pandas                 2.2.0
scipy                  1.12.0
sklearn                1.5.1
statsmodels            0.15.0
lightgbm               4.6.0
pvlib                  0.16.1
ruptures               v1.1.10
matplotlib             3.8.3
torch                  2.2.0+cpu
openpyxl               3.1.5
pyarrow                15.0.2
yaml                   6.0.1
on_colab               False
chronos-forecasting    not installed (PackageNotFoundError)
timesfm                not installed (PackageNotFoundError)
tabpfn-time-series     not installed (PackageNotFoundError)
tabpfn                 not installed (PackageNotFoundError)
cuda                   False
gpu                    None


## Data integrity
Every input file is compared with the committed SHA-256 manifest (D15).

In [3]:
res = subprocess.run([sys.executable, str(C.ROOT / 'tools' / 'make_data_manifest.py'), '--check'],
                     capture_output=True, text=True)
print(res.stdout[-2000:], res.stderr[-2000:])
manifest_ok = res.returncode == 0
assert manifest_ok, 'the data on disk differ from data/MANIFEST_sha256.csv'

169 files on disk, 169 in manifest, 0 differences
 


## Configuration snapshot
All constants of `src/pvcold/config.py` (guide §2–§3, D08–D12). Tuned values are added later to the frozen configuration (D21).

In [4]:
import yaml
cfg = {k: (str(v) if isinstance(v, Path) else v) for k, v in vars(C).items()
       if k.isupper() and not k.startswith('_')}
cfg = {k: (list(v) if isinstance(v, tuple) else v) for k, v in cfg.items()}
text = yaml.safe_dump(cfg, sort_keys=True, allow_unicode=True)
print(text[:1500])

CAMS_HKUST: C:\Users\Dell\Desktop\phd_ehtp\pv paper\data\external\cams\observed\cams-HKUST-21-23.csv
CAMS_SAFI: C:\Users\Dell\Desktop\phd_ehtp\pv paper\data\external\cams\observed\cams-safi-16-18.csv
CAMS_WH_PER_STEP_TO_W: 60.0
CLIP_THRESHOLD: 0.97
COMMON_TARGET_WINDOW:
- '12:00'
- '17:55'
CONNECTION_GAP_FLAG_DAYS: 30
DATA: C:\Users\Dell\Desktop\phd_ehtp\pv paper\data
DONOR_MIN_FULL_DAYS: 14
DRIFT_PROVISIONAL_BOUNDARY: '2022-11-01'
FIRST_TEST_DAY: '2021-10-05'
FROZEN_MINUTES: 60
GAIN_BOUNDS:
- 0.3
- 1.5
GAIN_FIT_MIN_GTILDE: 0.2
GAIN_FIT_MIN_K: 0.5
HORIZONS:
- 1
- 2
- 4
- 8
- 16
- 24
HUBER_DELTA: 0.05
INVERTER_FIRST_SLOT: 06:00
INVERTER_LAST_SLOT: '17:55'
INV_DIR: C:\Users\Dell\Desktop\phd_ehtp\pv paper\data\raw\hkust\Dataset\Time series
  dataset\PV generation dataset\PV stations with panel level optimizer\Inverter level
  dataset
ISSUE_MIN_DONORS: 8
ISSUE_MIN_GTILDE: 0.05
K_BOUNDS:
- 0.0
- 1.5
MCCLEAR_HKUST: C:\Users\Dell\Desktop\phd_ehtp\pv paper\data\external\cams\mcclear\HKUSTcsv.c

In [5]:
summary = {'notebook': 'NB0_setup_config', 'environment': env, 'manifest_ok': manifest_ok,
           'seed': C.SEED, 'runtime_s': round(time.time() - t0, 1)}
out = C.RESULTS / '.tmp_nb0_setup'
import shutil
if out.exists(): shutil.rmtree(out)
out.mkdir(parents=True)
(out / 'config_snapshot.yaml').write_text(text, encoding='utf-8')
(out / 'summary.json').write_text(json.dumps(summary, indent=2, default=str), encoding='utf-8')
final = C.RESULTS / 'nb0_setup'
if final.exists(): shutil.rmtree(final)
out.rename(final)
print('written:', sorted(p.name for p in final.iterdir()))
if ON_COLAB:
    drive.flush_and_unmount()

written: ['config_snapshot.yaml', 'summary.json']
